In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys
import inspect
import pandas as pd

# Add the project root to Python's path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [4]:
from src.data_pull import download_statcast, save_processed_data
from src.features import build_pitcher_games_table, add_rolling_features, identify_starters, add_pitch_mix_features, build_team_game_stats, add_team_rolling_features, add_historical_features

In [5]:
START_DATE = "2024-03-20"
END_DATE = "2024-09-30"

In [6]:
df = download_statcast(START_DATE, END_DATE)

This is a large query, it may take a moment to complete


c:\0Ian\Projects\Fantasy\.venv\Lib\site-packages\pybaseball\statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
100%|██████████| 195/195 [01:49<00:00,  1.78it/s]


Saving to C:\0Ian\Projects\Fantasy\data\raw\statcast_2024-03-20_2024-09-30.csv


In [46]:
pitcher_games = build_pitcher_games_table(df)

In [8]:
pitcher_games.head()

,game_date,game_pk,pitcher,player_name,pitches,strikeouts,avg_velocity,avg_spin,avg_break_x,avg_break_z,csw,whiffs,called_strikes,rest_days,max_times_through_order,CSW%
0,2024-03-20,745444,506433,"Darvish, Yu",73,3,87.384507,2468.422535,0.049718,0.315493,15,5,10,<NA>,2,0.205479
1,2024-03-20,745444,518489,"Brasier, Ryan",11,1,90.036364,2380.636364,-0.181818,0.860909,4,1,3,<NA>,1,0.363636
2,2024-03-20,745444,523260,"Kelly, Joe",8,0,95.4375,2335.375,-0.96375,0.6025,1,0,1,<NA>,1,0.125
3,2024-03-20,745444,543339,"Hudson, Daniel",19,1,91.647368,2438.052632,-0.397368,0.847895,6,4,2,<NA>,1,0.315789
4,2024-03-20,745444,593974,"Peralta, Wandy",21,1,91.521053,2176.526316,1.394211,0.762105,4,1,3,<NA>,1,0.190476


In [47]:
pitcher_games = add_rolling_features(pitcher_games)

In [10]:
pitcher_games.head()

,game_date,game_pk,pitcher,player_name,pitches,strikeouts,avg_velocity,avg_spin,avg_break_x,avg_break_z,...,velo_last3,spin_last3,csw_last3,pitches_last3,velo_last6,velo_trend,k_std_last5,pitches_last5,whiff_last3,csw_last5
3411,2024-04-19,744872,434378,"Verlander, Justin",78,4,87.25641,2393.128205,-0.26141,0.666795,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4197,2024-04-25,746887,434378,"Verlander, Justin",95,7,87.144211,2493.957895,-0.110211,0.625368,...,87.256410,2393.128205,0.179487,78.000000,NaN,NaN,NaN,78.00,4.000000,NaN
4868,2024-05-01,746399,434378,"Verlander, Justin",97,2,89.025773,2449.979381,-0.351649,0.790515,...,87.200310,2443.543050,0.237112,86.500000,NaN,NaN,2.121320,86.50,9.500000,0.237112
5467,2024-05-07,745752,434378,"Verlander, Justin",97,2,88.739175,2446.484536,0.012784,0.621856,...,87.808798,2445.688494,0.226803,90.000000,87.808798,0.000000,2.516611,90.00,9.000000,0.226803
6066,2024-05-12,746476,434378,"Verlander, Justin",95,8,87.48,2466.589474,-0.093579,0.718737,...,88.303053,2463.473937,0.228830,96.333333,88.041392,0.261661,2.362908,91.75,9.333333,0.216494


In [48]:
starters = identify_starters(df)

starters.head()

,game_pk,pitcher,is_starter
0,744795,547179,True
11,744795,687792,True
12,744796,571578,True
15,744796,608379,True
22,744797,592866,True


In [49]:
pitcher_games = pitcher_games.merge(starters, on=["game_pk", "pitcher"], how="left")

pitcher_games["is_starter"] = (pitcher_games["is_starter"].fillna(False))

In [50]:
pitcher_games = add_pitch_mix_features(df, pitcher_games)

In [14]:
pitcher_games.shape

(21659, 46)

In [51]:
pitcher_games = pitcher_games[pitcher_games["is_starter"] == True].copy()

In [52]:
team_games = build_team_game_stats(df)

team_games.head()

,game_date,game_pk,team,opponent,runs,plate_appearances,hits,home_runs,strikeouts,walks,iso,k_rate,bb_rate,hr_rate
0,2024-03-20,745444,LAD,SD,5,79,11,0,13,13,0.0,0.164557,0.164557,0.000000
1,2024-03-20,745444,SD,LAD,2,79,11,0,13,13,0.0,0.164557,0.164557,0.000000
2,2024-03-20,747878,CIN,TEX,1,79,17,2,25,4,0.088608,0.316456,0.050633,0.025316
3,2024-03-20,747878,TEX,CIN,8,79,17,2,25,4,0.088608,0.316456,0.050633,0.025316
4,2024-03-20,747879,CIN,CWS,3,73,13,2,19,6,0.136986,0.260274,0.082192,0.027397


In [53]:
team_games = add_team_rolling_features(team_games)

team_games.head()

,game_date,game_pk,team,opponent,runs,plate_appearances,hits,home_runs,strikeouts,walks,iso,k_rate,bb_rate,hr_rate,runs_last14,k_rate_last14,bb_rate_last14,hr_rate_last14,iso_last14
10,2024-03-20,747882,ATH,CHC,3,61,7,3,20,2,0.163934,0.327869,0.032787,0.049180,NaN,NaN,NaN,NaN,NaN
62,2024-03-22,747853,ATH,CWS,2,76,19,2,17,8,0.131579,0.223684,0.105263,0.026316,NaN,NaN,NaN,NaN,NaN
66,2024-03-22,747855,ATH,CIN,5,81,17,3,23,12,0.197531,0.283951,0.148148,0.037037,NaN,NaN,NaN,NaN,NaN
90,2024-03-23,747836,ATH,LAA,11,81,22,3,18,8,0.209877,0.222222,0.098765,0.037037,3.333333,0.278501,0.095399,0.037511,0.164348
140,2024-03-25,747807,ATH,SF,1,71,10,3,23,8,0.140845,0.323944,0.112676,0.042254,5.250000,0.264431,0.096241,0.037393,0.175730


In [54]:
pitcher_opponents = (
    df.groupby(["game_pk", "pitcher"])
    .agg(
        game_date=("game_date", "first"),
        inning_topbot=("inning_topbot", "first"),
        home_team=("home_team", "first"),
        away_team=("away_team", "first")
    )
    .reset_index()
)


pitcher_opponents["opponent"] = pitcher_opponents.apply(lambda row: row["home_team"] if row["inning_topbot"] == "Bot" else row["away_team"], axis=1)

pitcher_opponents.head()

,game_pk,pitcher,game_date,inning_topbot,home_team,away_team,opponent
0,744795,547179,2024-09-25,Bot,WSH,KC,WSH
1,744795,606930,2024-09-25,Top,WSH,KC,KC
2,744795,663432,2024-09-25,Top,WSH,KC,KC
3,744795,663738,2024-09-25,Bot,WSH,KC,WSH
4,744795,668674,2024-09-25,Bot,WSH,KC,WSH


In [55]:
pitcher_games = pitcher_games.merge(pitcher_opponents[["game_pk", "pitcher", "opponent"]], on=["game_pk", "pitcher"], how="left")

In [56]:
opponent_features = team_games[["game_date", "team", "runs_last14", "k_rate_last14", "bb_rate_last14", "hr_rate_last14", "iso_last14"]].copy()


opponent_features = opponent_features.rename(
    columns={
        "team": "opponent",
        "runs_last14": "opp_runs_last14",
        "k_rate_last14": "opp_k_rate_last14",
        "bb_rate_last14": "opp_bb_rate_last14",
        "hr_rate_last14": "opp_hr_rate_last14",
        "iso_last14": "opp_iso_last14"
    }
)

In [57]:
pitcher_games["game_date"] = pd.to_datetime(
    pitcher_games["game_date"]
)

opponent_features["game_date"] = pd.to_datetime(
    opponent_features["game_date"]
)

In [58]:
pitcher_games = pitcher_games.merge(opponent_features, on=["game_date", "opponent"], how="left")

In [59]:
game_locations = (df.groupby("game_pk").agg(home_team=("home_team", "first"), away_team=("away_team", "first")).reset_index())

pitcher_games = pitcher_games.merge(game_locations, on="game_pk", how="left")

In [60]:
park_factors = pd.DataFrame({
    "team": [
        "AZ","ATL","BAL","BOS","CHC","CWS","CIN","CLE",
        "COL","DET","HOU","KC","LAA","LAD","MIA","MIL",
        "MIN","NYM","NYY","ATH","PHI","PIT","SD","SEA",
        "SF","STL","TB","TEX","TOR","WSH"
    ],

    # FanGraphs 2024 Basic park factors
    "park_run_factor": [
        100,100,98,102,96,98,102,100,
        114,103,100,103,101,98,102,98,
        103,98,99,97,102,102,97,92,
        96,99,98,97,100,100
    ],

    # FanGraphs 2024 HR park factors
    "park_hr_factor": [
        91,99,99,98,98,105,114,98,
        107,96,102,95,105,110,97,104,
        99,99,104,90,105,93,101,96,
        91,94,96,102,103,100
    ]
})

In [61]:
park_factors["park_run_factor"] = (park_factors["park_run_factor"] / 100)

park_factors["park_hr_factor"] = (park_factors["park_hr_factor"] / 100)

In [62]:
pitcher_games = pitcher_games.merge(park_factors, left_on="home_team", right_on="team", how="left")

pitcher_games = pitcher_games.drop(columns=["team"])

In [63]:
historical_cols = ["avg_velocity", "avg_spin", "avg_break_x", "avg_break_z", "csw", "whiffs", "called_strikes"]

In [64]:
historical_cols += [col for col in pitcher_games.columns if "pitch_" in col]

In [65]:
pitcher_games_whistoric = add_historical_features(pitcher_games,historical_cols)

In [71]:
save_processed_data(pitcher_games_whistoric)

Saving as final_pitcher_modeling_table.csv
